# LeetCode 2038 — Remove Colored Pieces if Both Neighbors are the Same Color bằng Python

Link đề: https://leetcode.com/problems/remove-colored-pieces-if-both-neighbors-are-the-same-color/ · Độ khó: **Medium** · Chủ đề: Math, String, Greedy, Game Theory

Skeleton LeetCode:

```python
class Solution:
    def winnerOfGame(self, colors: str) -> bool:
        ...
```

## 1. Mục tiêu học

1. Đọc một bài **trò chơi hai người** và nhận ra nó **không cần minimax**: nước đi của hai người **độc lập** với nhau, nên chỉ cần **đếm số nước đi** mỗi người có.
2. Chứng minh tính độc lập: xóa một `'A'` giữa ba `'A'` không tạo ra, cũng không phá hủy bất kỳ nước đi nào của `'B'`, và chỉ làm giảm đúng **1** số nước đi của chính Alice.
3. Đếm số bộ ba liên tiếp cùng màu `XXX` trong một lượt `O(n)`; liên hệ với độ dài loạt (run) `L` cho `max(0, L − 2)` nước.
4. Hiểu quy tắc "đi trước": Alice thắng ⇔ số nước của Alice **lớn hơn hẳn** số nước của Bob.
5. Viết **minimax có ghi nhớ** trên chuỗi ngắn làm brute force để kiểm chứng công thức.

## 2. Đề bài & ví dụ

Có `n` quân xếp thành một hàng, mỗi quân màu `'A'` hoặc `'B'`, cho bởi chuỗi `colors`. Alice và Bob lần lượt xóa quân, **Alice đi trước**:
- Alice chỉ được xóa một quân `'A'` nếu **cả hai hàng xóm** của nó cũng là `'A'`; không được xóa `'B'`.
- Bob chỉ được xóa một quân `'B'` nếu **cả hai hàng xóm** của nó cũng là `'B'`; không được xóa `'A'`.
- Không ai được xóa quân ở **hai đầu** hàng.
- Người không có nước đi ở lượt của mình **thua**.

Giả sử cả hai chơi tối ưu, trả `true` nếu Alice thắng, `false` nếu Bob thắng.

**Ví dụ 1:** `"AAABABB"` → `true` (Alice xóa `'A'` thứ hai; Bob không còn nước).

**Ví dụ 2:** `"AA"` → `false` (Alice không có nước ngay từ đầu).

**Ví dụ 3:** `"ABBBBBBBAAA"` → `false` (Alice có 1 nước, Bob có 5 nước; Alice hết nước trước).

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; chỉ gồm `'A'`, `'B'`.

In [1]:
import random
import re
import time
from functools import lru_cache
from itertools import groupby, product
from typing import List

examples = [("AAABABB", True), ("AA", False), ("ABBBBBBBAAA", False)]
for s, exp in examples:
    print(f"{s!r} -> kỳ vọng {exp}")

'AAABABB' -> kỳ vọng True
'AA' -> kỳ vọng False
'ABBBBBBBAAA' -> kỳ vọng False


## 3. Kiến thức nền

### 3.1 Trò chơi tổ hợp và minimax

Một trạng thái là **thắng** cho người đang đi nếu tồn tại một nước dẫn tới trạng thái **thua** cho đối thủ; là **thua** nếu không có nước nào hoặc mọi nước đều dẫn tới trạng thái thắng cho đối thủ. Duyệt toàn bộ cây trò chơi (có ghi nhớ) cho đáp án chắc chắn — nhưng số trạng thái tăng theo hàm mũ.

### 3.2 Loạt (run) ký tự giống nhau

`itertools.groupby` chia chuỗi thành các loạt ký tự liên tiếp giống nhau, ví dụ `"AAABABB"` → `AAA | B | A | BB`.

In [2]:
s = "AAABABB"
print([(ch, len(list(g))) for ch, g in groupby(s)])

[('A', 3), ('B', 1), ('A', 1), ('B', 2)]


### 3.3 Quân "xóa được" trong một loạt

Trong loạt `'A'` dài `L`, quân ở vị trí `k` (`0 ≤ k < L`) có hai hàng xóm `'A'` ⇔ `0 < k < L − 1`. Hai quân ở đầu loạt có một hàng xóm khác màu (hoặc là mép hàng). Vậy loạt dài `L` có `max(0, L − 2)` quân xóa được **ở thời điểm hiện tại**.

### 3.4 Đếm bộ ba liên tiếp

Số vị trí `i` (`1 ≤ i ≤ n−2`) với `colors[i−1] = colors[i] = colors[i+1] = 'A'` chính là tổng `max(0, L − 2)` trên các loạt `'A'`.

In [3]:
def triples(s, ch):
    return sum(1 for i in range(1, len(s) - 1) if s[i - 1] == s[i] == s[i + 1] == ch)

for s, _ in examples:
    runs = [(c, len(list(g))) for c, g in groupby(s)]
    print(f"{s:<12} bộ ba A={triples(s, 'A')} B={triples(s, 'B')} | theo loạt A={sum(max(0, L - 2) for c, L in runs if c == 'A')}")

AAABABB      bộ ba A=1 B=0 | theo loạt A=1
AA           bộ ba A=0 B=0 | theo loạt A=0
ABBBBBBBAAA  bộ ba A=1 B=5 | theo loạt A=1


### 3.5 Lượt thứ `t` là của ai

Đánh số lượt từ 1: lượt lẻ của Alice, lượt chẵn của Bob. Trước lượt `2k + 1`, mỗi người đã đi `k` nước; trước lượt `2k + 2`, Alice đã đi `k + 1`, Bob `k`. Công thức này dùng trong chứng minh ở mục 6 và 10.

In [4]:
for t in range(1, 8):
    who = "Alice" if t % 2 else "Bob"
    done_a, done_b = t // 2, (t - 1) // 2
    print(f"lượt {t}: {who:<5} | trước lượt này Alice đã đi {done_a}, Bob đã đi {done_b}")

lượt 1: Alice | trước lượt này Alice đã đi 0, Bob đã đi 0
lượt 2: Bob   | trước lượt này Alice đã đi 1, Bob đã đi 0
lượt 3: Alice | trước lượt này Alice đã đi 1, Bob đã đi 1
lượt 4: Bob   | trước lượt này Alice đã đi 2, Bob đã đi 1
lượt 5: Alice | trước lượt này Alice đã đi 2, Bob đã đi 2
lượt 6: Bob   | trước lượt này Alice đã đi 3, Bob đã đi 2
lượt 7: Alice | trước lượt này Alice đã đi 3, Bob đã đi 3


## 4. Ý tưởng nền tảng / trực giác

Thử chơi tay **Ví dụ 3** `"ABBBBBBBAAA"`: loạt `B` dài 7 cho Bob `5` nước; loạt `AAA` cho Alice `1` nước.

| lượt | người | trạng thái trước | nước | Alice còn | Bob còn |
|---|---|---|---|---|---|
| 1 | Alice | `ABBBBBBBAAA` | xóa `A` giữa `AAA` | 0 | 5 |
| 2 | Bob | `ABBBBBBBAA` | xóa một `B` giữa | 0 | 4 |
| 3 | Alice | `ABBBBBBAA` | **không có nước** ⇒ thua | | |

Quan sát quan trọng:
1. Xóa một `'A'` khỏi loạt `'A'` dài `L` biến nó thành loạt dài `L − 1` — **chỉ** làm giảm số nước của Alice đúng 1.
2. Vì quân bị xóa nằm **giữa** hai `'A'`, các `'B'` không hề "chạm" nhau sau khi xóa ⇒ **không** tạo hay mất nước nào của Bob. Tương tự cho Bob.

Vậy mỗi người có một "kho" nước đi cố định từ đầu: Alice có `a`, Bob có `b`, và không ai ảnh hưởng được kho của người kia. Trò chơi chỉ là hai người lần lượt rút từ kho của mình; ai hết trước khi tới lượt thì thua.

In [5]:
def show_game(s):
    a, b = triples(s, "A"), triples(s, "B")
    turn, k = "Alice", 0
    while True:
        k += 1
        if turn == "Alice":
            if a == 0: print(f"lượt {k}: Alice hết nước -> Bob thắng"); return False
            a -= 1
        else:
            if b == 0: print(f"lượt {k}: Bob hết nước -> Alice thắng"); return True
            b -= 1
        print(f"lượt {k}: {turn} đi, kho còn A={a}, B={b}")
        turn = "Bob" if turn == "Alice" else "Alice"

for s, exp in examples:
    print("==", s); assert show_game(s) == exp

== AAABABB
lượt 1: Alice đi, kho còn A=0, B=0
lượt 2: Bob hết nước -> Alice thắng
== AA
lượt 1: Alice hết nước -> Bob thắng
== ABBBBBBBAAA
lượt 1: Alice đi, kho còn A=0, B=5
lượt 2: Bob đi, kho còn A=0, B=4
lượt 3: Alice hết nước -> Bob thắng


## 5. Cách 1 — Brute Force / baseline: minimax có ghi nhớ trên chuỗi

Trạng thái = (chuỗi hiện tại, lượt ai). Thử mọi nước hợp lệ; người đi thắng nếu có nước dẫn tới trạng thái thua của đối thủ.

In [6]:
def winner_minimax(colors):
    @lru_cache(maxsize=None)
    def win(s, ch):                                   # người cầm màu ch đang đi, có thắng không?
        other = "B" if ch == "A" else "A"
        for i in range(1, len(s) - 1):
            if s[i - 1] == s[i] == s[i + 1] == ch:
                if not win(s[:i] + s[i + 1:], other):
                    return True
        return False
    return win(colors, "A")

for s, exp in examples:
    got = winner_minimax(s); print(s, "->", got, "kỳ vọng", exp); assert got == exp

AAABABB -> True kỳ vọng True
AA -> False kỳ vọng False
ABBBBBBBAAA -> False kỳ vọng False


**Độ phức tạp:** số trạng thái là số chuỗi con nhận được qua các phép xóa — tăng theo hàm mũ với độ dài loạt; mỗi trạng thái `O(n²)` (thử và cắt chuỗi). Chỉ dùng với `n ≲ 14`.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (độc lập):** tập nước đi của Alice và của Bob không ảnh hưởng nhau. Mỗi nước của Alice làm kho của Alice giảm đúng 1, kho của Bob giữ nguyên (và ngược lại).

**Insight 2 (chơi tối ưu không có ý nghĩa):** vì mọi nước của Alice đều có cùng tác dụng (giảm `a` đi 1), **không có lựa chọn chiến lược** nào. Kết quả hoàn toàn xác định bởi `(a, b)`.

**Insight 3 (ai thua):** lượt thứ `2k + 1` là của Alice (đã dùng `k` nước), lượt `2k + 2` của Bob (đã dùng `k`). Alice thua ở lượt `2a + 1` (cần nước thứ `a + 1`); Bob thua ở lượt `2b + 2`. Alice thắng ⇔ `2b + 2 < 2a + 1` ⇔ `b < a`.

**Đáp án:** `a > b`, với `a`, `b` là số bộ ba `AAA`, `BBB` liên tiếp.

### 6.1 Minh họa tính độc lập trên một chuỗi cụ thể

Với `"AAAABBBAA"` (ví dụ tự dựng, đối chiếu lại đề gốc): thử **mọi** nước của Alice và in số nước còn lại của mỗi người sau nước đó. Kho của Bob không đổi, kho của Alice luôn giảm đúng 1.

In [7]:
s = "AAAABBBAA"
a0, b0 = triples(s, "A"), triples(s, "B")
print(f"ban đầu: a={a0}, b={b0}")
for i in range(1, len(s) - 1):
    if s[i - 1] == s[i] == s[i + 1] == "A":
        t = s[:i] + s[i + 1:]
        print(f"Alice xóa vị trí {i}: {t} -> a={triples(t, 'A')}, b={triples(t, 'B')}")
        assert triples(t, "A") == a0 - 1 and triples(t, "B") == b0

ban đầu: a=2, b=1
Alice xóa vị trí 1: AAABBBAA -> a=1, b=1
Alice xóa vị trí 2: AAABBBAA -> a=1, b=1


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đếm theo loạt bằng `groupby`

In [8]:
def winner_runs(colors):
    a = b = 0
    for ch, g in groupby(colors):
        L = len(list(g))
        if L >= 3:
            if ch == "A": a += L - 2
            else: b += L - 2
    return a > b

print([winner_runs(s) for s, _ in examples])

[True, False, False]


**Độ phức tạp Cách 2:** `O(n)` thời gian, `O(L)` bộ nhớ tạm cho mỗi loạt (`len(list(g))`; có thể thay bằng `sum(1 for _ in g)` để `O(1)`).

### 7.2 Cách 3 — Một lượt đếm bộ ba (bản nộp)

In [9]:
class Solution:
    def winnerOfGame(self, colors: str) -> bool:
        a = b = 0
        for i in range(1, len(colors) - 1):
            c = colors[i]
            if colors[i - 1] == c == colors[i + 1]:
                if c == "A":
                    a += 1
                else:
                    b += 1
        return a > b

print([Solution().winnerOfGame(s) for s, _ in examples])

[True, False, False]


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(1)` bộ nhớ.

### 7.3 Cách 4 — Đếm chồng lấn bằng regex lookahead / `zip`

Biểu thức `(?=AAA)` khớp **mọi vị trí** bắt đầu một bộ ba, kể cả chồng lấn (lookahead không "ăn" ký tự). Cách tương đương không regex: `zip(s, s[1:], s[2:])`.

In [10]:
def winner_regex(colors):
    return len(re.findall(r"(?=AAA)", colors)) > len(re.findall(r"(?=BBB)", colors))

def winner_zip(colors):
    a = sum(1 for x, y, z in zip(colors, colors[1:], colors[2:]) if x == y == z == "A")
    b = sum(1 for x, y, z in zip(colors, colors[1:], colors[2:]) if x == y == z == "B")
    return a > b

print([winner_regex(s) for s, _ in examples], [winner_zip(s) for s, _ in examples])
print("không lookahead:", re.findall("AAA", "AAAAA"), "| có lookahead:", len(re.findall("(?=AAA)", "AAAAA")))

[True, False, False] [True, False, False]
không lookahead: ['AAA'] | có lookahead: 3


`re.findall("AAA", "AAAAA")` chỉ cho **1** kết quả vì khớp không chồng lấn; lookahead cho đúng `3 = 5 − 2`.

**Độ phức tạp Cách 4:** `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1** `"AAABABB"`:

In [11]:
def trace(colors):
    a = b = 0
    for i in range(1, len(colors) - 1):
        w = colors[i - 1:i + 2]
        hit = w[0] == w[1] == w[2]
        if hit:
            if w[1] == "A": a += 1
            else: b += 1
        print(f"i={i}: cửa sổ {w} {'-> bộ ba ' + w[1] if hit else '':<12} a={a}, b={b}")
    print("=> a > b ?", a > b)
    return a > b

assert trace("AAABABB") is True

i=1: cửa sổ AAA -> bộ ba A   a=1, b=0
i=2: cửa sổ AAB              a=1, b=0
i=3: cửa sổ ABA              a=1, b=0
i=4: cửa sổ BAB              a=1, b=0
i=5: cửa sổ ABB              a=1, b=0
=> a > b ? True


| i | cửa sổ | bộ ba? | a | b |
|---|---|---|---|---|
| 1 | `AAA` | A | 1 | 0 |
| 2 | `AAB` | — | 1 | 0 |
| 3 | `ABA` | — | 1 | 0 |
| 4 | `BAB` | — | 1 | 0 |
| 5 | `ABB` | — | 1 | 0 |

`a = 1 > b = 0` ⇒ Alice thắng. Trace thêm **Ví dụ 3** để thấy `b = 5`:

In [12]:
assert trace("ABBBBBBBAAA") is False

i=1: cửa sổ ABB              a=0, b=0
i=2: cửa sổ BBB -> bộ ba B   a=0, b=1
i=3: cửa sổ BBB -> bộ ba B   a=0, b=2
i=4: cửa sổ BBB -> bộ ba B   a=0, b=3
i=5: cửa sổ BBB -> bộ ba B   a=0, b=4
i=6: cửa sổ BBB -> bộ ba B   a=0, b=5
i=7: cửa sổ BBA              a=0, b=5
i=8: cửa sổ BAA              a=0, b=5
i=9: cửa sổ AAA -> bộ ba A   a=1, b=5
=> a > b ? False


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — một lượt đếm bộ ba, `O(n)` / `O(1)`.

In [13]:
class Solution:
    def winnerOfGame(self, colors: str) -> bool:
        a = b = 0
        for i in range(1, len(colors) - 1):
            c = colors[i]
            if colors[i - 1] == c == colors[i + 1]:
                if c == "A":
                    a += 1
                else:
                    b += 1
        return a > b

for s, exp in examples:
    got = Solution().winnerOfGame(s); print(got, exp); assert got == exp

True True
False False
False False


## 10. Vì sao thuật toán đúng

Gọi `a(s)`, `b(s)` là số bộ ba `AAA`, `BBB` liên tiếp trong chuỗi `s`.

**Bổ đề 1.** Nếu Alice xóa `s[i]` với `s[i−1] = s[i] = s[i+1] = 'A'`, chuỗi mới `t` có `a(t) = a(s) − 1` và `b(t) = b(s)`.

*Chứng minh.* `s[i]` nằm trong một loạt `'A'` dài `L ≥ 3`. Xóa nó cho loạt dài `L − 1`; các loạt khác không đổi, và hai hàng xóm của `s[i]` đều là `'A'` nên không có hai loạt nào bị nối lại với nhau. Số bộ ba của loạt `'A'` đổi từ `L − 2` thành `L − 3`; mọi loạt `'B'` giữ nguyên ⇒ `b` không đổi. ∎ (Đối xứng cho Bob.)

**Bổ đề 2.** Người đang đi có nước ⇔ kho của họ `> 0` (theo định nghĩa bộ ba).

**Định lý.** Alice thắng ⇔ `a(s) > b(s)`.

*Chứng minh.* Theo Bổ đề 1, sau `k` nước của Alice và `k'` nước của Bob, kho là `(a − k, b − k')`, bất kể chọn nước nào — không có lựa chọn chiến lược. Lượt của Alice khi cả hai đã đi `k` nước: Alice thua nếu `a − k = 0`. Lượt của Bob khi Alice đã đi `k + 1`, Bob `k`: Bob thua nếu `b − k = 0`. Người thua là người chạm 0 trước: Alice thua ở lượt `2a + 1`, Bob ở lượt `2b + 2`. Alice thắng ⇔ `2b + 2 < 2a + 1` ⇔ `b < a` (số nguyên). ∎

### 10.1 Kiểm chứng thực nghiệm: minimax = công thức trên **mọi** chuỗi ngắn

Duyệt hết `2¹ + … + 2¹²` chuỗi và so minimax với `a > b`. Đồng thời kiểm tra Bổ đề 1 trên mọi nước đi của mọi chuỗi độ dài `≤ 10`.

In [14]:
cnt = 0
for n in range(1, 13):
    for bits in product("AB", repeat=n):
        s = "".join(bits)
        assert winner_minimax(s) == Solution().winnerOfGame(s), s
        cnt += 1
for n in range(3, 11):
    for bits in product("AB", repeat=n):
        s = "".join(bits); a0, b0 = triples(s, "A"), triples(s, "B")
        for i in range(1, n - 1):
            if s[i - 1] == s[i] == s[i + 1]:
                t = s[:i] + s[i + 1:]
                if s[i] == "A": assert (triples(t, "A"), triples(t, "B")) == (a0 - 1, b0)
                else: assert (triples(t, "A"), triples(t, "B")) == (a0, b0 - 1)
print(f"minimax = công thức trên {cnt} chuỗi; Bổ đề 1 đúng với mọi nước đi (n ≤ 10)")

minimax = công thức trên 8190 chuỗi; Bổ đề 1 đúng với mọi nước đi (n ≤ 10)


### 10.2 Phản ví dụ: khi luật đổi, tính độc lập biến mất

Giả sử luật là "được xóa quân màu của mình nếu **hai hàng xóm cùng màu với nhau**" (không cần cùng màu với quân bị xóa). Khi Alice xóa một `'A'` nằm giữa hai `'B'`, hai `'B'` nối lại và Bob **có thêm** nước. Lúc đó đếm kho không còn đúng — cần minimax thật sự. Đoạn code dưới tìm các chuỗi mà "so sánh kho" cho kết quả khác minimax dưới luật mới, để thấy rõ Bổ đề 1 là chìa khóa của bài gốc.

In [15]:
def win_variant(colors):
    @lru_cache(maxsize=None)
    def win(s, ch):
        other = "B" if ch == "A" else "A"
        for i in range(1, len(s) - 1):
            if s[i] == ch and s[i - 1] == s[i + 1] and not win(s[:i] + s[i + 1:], other):
                return True
        return False
    return win(colors, "A")

def stock_variant(colors):
    a = sum(1 for i in range(1, len(colors) - 1) if colors[i] == "A" and colors[i - 1] == colors[i + 1])
    b = sum(1 for i in range(1, len(colors) - 1) if colors[i] == "B" and colors[i - 1] == colors[i + 1])
    return a > b

diff = [x for n in range(3, 9) for x in map("".join, product("AB", repeat=n)) if win_variant(x) != stock_variant(x)]
print(f"Luật biến thể: {len(diff)} chuỗi (độ dài ≤ 8) mà đếm kho sai, ví dụ {diff[:4]}")

Luật biến thể: 117 chuỗi (độ dài ≤ 8) mà đếm kho sai, ví dụ ['ABAB', 'BABA', 'BABB', 'BBAB']


## 11. Phân tích Time / Space Complexity

Gọi `n = len(colors) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Minimax ghi nhớ | mũ | mũ |
| 2. Theo loạt (`groupby`) | `O(n)` | `O(1)`–`O(L)` |
| 3. Một lượt đếm bộ ba | `O(n)` | `O(1)` |
| 4. Regex lookahead / `zip` | `O(n)` | `O(n)` (lát cắt) |

In [16]:
rng = random.Random(2038)
big = "".join(rng.choice("AB") for _ in range(10 ** 5))
runs = "A" * 50000 + "B" * 50000
for name, f in [("runs", winner_runs), ("submit", Solution().winnerOfGame), ("regex", winner_regex), ("zip", winner_zip)]:
    t = time.perf_counter(); r1 = f(big); r2 = f(runs); print(f"{name:<7}: {r1}, {r2} ({(time.perf_counter() - t) * 1000:.0f} ms cho 2 chuỗi 1e5)")
for k in (2, 4, 6, 7):
    s = "AAAAB" * k + "B" * (2 * k + 3)  # k loạt A độc lập + một loạt B dài: Alice thua, minimax phải xét mọi thứ tự
    t = time.perf_counter(); r = winner_minimax(s); print(f"minimax k={k} (n={len(s)}): {r} ({(time.perf_counter() - t) * 1000:.1f} ms)")

runs   : False, False (7 ms cho 2 chuỗi 1e5)
submit : False, False (12 ms cho 2 chuỗi 1e5)
regex  : False, False (11 ms cho 2 chuỗi 1e5)
zip    : False, False (15 ms cho 2 chuỗi 1e5)
minimax k=2 (n=17): False (0.0 ms)
minimax k=4 (n=31): False (0.2 ms)
minimax k=6 (n=45): False (2.9 ms)
minimax k=7 (n=52): False (9.0 ms)


## 12. Bộ test

In [17]:
ALL = {"runs": winner_runs, "submit": Solution().winnerOfGame, "regex": winner_regex, "zip": winner_zip}
def run_all(s, brute=True):
    res = {k: f(s) for k, f in ALL.items()}
    if brute:
        res["minimax"] = winner_minimax(s)
    assert len(set(res.values())) == 1, (s, res)
    return res["submit"]

for s, exp in examples:
    assert run_all(s) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng minimax trong run_all
edge = [("A", False), ("B", False), ("AAA", True), ("BBB", False), ("AAABBB", False), ("AAAABBB", True),
        ("ABABAB", False), ("AAAAABBBB", True), ("BBBBAAAA", False), ("AAABBBAAA", True)]
for s, exp in edge:
    assert run_all(s) == exp, (s, run_all(s))
rng = random.Random(20380)
for _ in range(400):
    run_all("".join(rng.choice("AB") for _ in range(rng.randint(1, 13))))
for _ in range(20):
    n = rng.randint(1, 10 ** 5); p = rng.random()
    run_all("".join("A" if rng.random() < p else "B" for _ in range(n)), brute=False)
print("Ví dụ, 10 edge case, 400 chuỗi ngắn (minimax), 20 chuỗi dài: OK")

Ví dụ, 10 edge case, 400 chuỗi ngắn (minimax), 20 chuỗi dài: OK


### 12.1 Kiểm tra tính chất

- Đảo ngược chuỗi không đổi đáp án (bộ ba đối xứng).
- Đổi màu `A ↔ B`: kết quả mới là `b > a`; nếu `a ≠ b` thì đúng một trong hai bản Alice thắng; nếu `a = b` thì cả hai đều Bob thắng.
- Thêm một `'A'` vào **giữa** một loạt `'A'` dài `≥ 2` chỉ có thể giúp Alice.

In [18]:
rng = random.Random(121)
f = Solution().winnerOfGame
sw = str.maketrans("AB", "BA")
for _ in range(3000):
    s = "".join(rng.choice("AB") for _ in range(rng.randint(1, 30)))
    a, b = triples(s, "A"), triples(s, "B")
    assert f(s[::-1]) == f(s)
    if a != b: assert f(s) != f(s.translate(sw))
    else: assert not f(s) and not f(s.translate(sw))
    pos = [i for i in range(1, len(s)) if s[i - 1] == s[i] == "A"]
    if pos:
        i = rng.choice(pos); t = s[:i] + "A" + s[i:]
        assert triples(t, "A") == a + 1 and (f(t) or not f(s))
print("3 tính chất đúng trên 3000 chuỗi")

3 tính chất đúng trên 3000 chuỗi


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `a >= b` (quên lợi thế/bất lợi của người đi trước)

In [19]:
def wrong_ge(colors):
    return triples(colors, "A") >= triples(colors, "B")                 # SAI

for s in ["AA", "AAABBB"]:          # "AAABBB" là ví dụ tự dựng (đối chiếu lại đề gốc)
    print(f"{s}: sai {wrong_ge(s)} | đúng {Solution().winnerOfGame(s)} | minimax {winner_minimax(s)}")

AA: sai True | đúng False | minimax False
AAABBB: sai True | đúng False | minimax False


**Vì sao sai:** khi `a = b`, Alice (đi trước) dùng hết kho trước ⇒ đến lượt Alice thì không còn nước. Cả `"AA"` (`0 = 0`) lẫn `"AAABBB"` (`1 = 1`) đều Bob thắng. **Sửa:** `a > b`.

### Lỗi 2 — Đếm `L − 1` (hoặc `L`) nước cho loạt dài `L`

In [20]:
def wrong_L_minus_1(colors):
    a = b = 0
    for ch, g in groupby(colors):
        L = len(list(g))
        if ch == "A": a += max(0, L - 1)
        else: b += max(0, L - 1)                                         # SAI: phải L - 2
    return a > b

for s, exp in examples:
    print(f"{s}: sai {wrong_L_minus_1(s)} | đúng {exp}")

AAABABB: sai True | đúng True
AA: sai True | đúng False
ABBBBBBBAAA: sai False | đúng False


**Vì sao sai:** hai quân ở **hai đầu** loạt đều có một hàng xóm khác màu (hoặc là mép), nên không xóa được. Ở `"AA"`, cách sai cho Alice 1 nước. **Sửa:** `max(0, L − 2)`.

### Lỗi 3 — Đếm bộ ba **không chồng lấn**

In [21]:
def wrong_nonoverlap(colors):
    return colors.count("AAA") > colors.count("BBB")                    # SAI: str.count không chồng lấn

s = "AAAAABBBB"     # ví dụ tự dựng (đối chiếu lại đề gốc)
print(f"{s}: count AAA={s.count('AAA')}, BBB={s.count('BBB')} -> sai {wrong_nonoverlap(s)} | đúng {Solution().winnerOfGame(s)} | minimax {winner_minimax(s)}")

AAAAABBBB: count AAA=1, BBB=1 -> sai False | đúng True | minimax True


**Vì sao sai:** `"AAAAA".count("AAA") = 1` nhưng loạt dài 5 cho **3** nước. **Sửa:** đếm từng vị trí (Cách 3) hoặc lookahead `(?=AAA)`.

### Lỗi 4 — Nghĩ rằng Alice nên xóa để "chặn" Bob (và viết mô phỏng tham lam phức tạp)

In [22]:
def greedy_block(colors):
    # "chiến lược" tham lam: Alice luôn xóa quân A cạnh B nhất (vô nghĩa vì không ảnh hưởng Bob)
    s, turn = colors, "A"
    while True:
        moves = [i for i in range(1, len(s) - 1) if s[i - 1] == s[i] == s[i + 1] == turn]
        if not moves:
            return turn == "B"
        i = min(moves, key=lambda k: min(abs(k - j) for j in range(len(s)) if s[j] != turn) if any(c != turn for c in s) else 0)
        s = s[:i] + s[i + 1:]; turn = "B" if turn == "A" else "A"

rng = random.Random(4); same = 0
for _ in range(300):
    s = "".join(rng.choice("AB") for _ in range(rng.randint(1, 40)))
    same += greedy_block(s) == Solution().winnerOfGame(s)
print(f"Tham lam 'chặn' cho cùng kết quả trên {same}/300 chuỗi — chiến lược không có tác dụng, chỉ tốn O(n²)")

Tham lam 'chặn' cho cùng kết quả trên 300/300 chuỗi — chiến lược không có tác dụng, chỉ tốn O(n²)


**Vì sao không tốt:** kết quả vẫn đúng (vì mọi nước đều tương đương — Bổ đề 1), nhưng mô phỏng với cắt chuỗi `O(n)` mỗi nước ⇒ `O(n²)` và code phức tạp vô ích. **Sửa:** chỉ đếm.

### Lỗi 5 — Vòng lặp chạy hết `range(len(s))` và truy cập `s[i + 1]`

In [23]:
def wrong_range(colors):
    a = b = 0
    try:
        for i in range(len(colors)):                                       # SAI: i = 0 dùng s[-1], i = n-1 tràn
            if colors[i - 1] == colors[i] == colors[i + 1]:
                if colors[i] == "A": a += 1
                else: b += 1
    except IndexError as e:
        return f"IndexError: {e}"
    return a > b

print("AAABABB ->", wrong_range("AAABABB"))
print("AABA (tự dựng, đối chiếu lại đề gốc) ->", wrong_range("AABA"), "| đúng:", Solution().winnerOfGame("AABA"), "| minimax:", winner_minimax("AABA"))

AAABABB -> IndexError: string index out of range
AABA (tự dựng, đối chiếu lại đề gốc) -> True | đúng: False | minimax: False


**Vì sao sai:** `colors[-1]` trong Python là ký tự **cuối** (không báo lỗi!) nên ở `i = 0` chuỗi `"AABA"` tạo bộ ba "vòng" giả `A|AA`; ở `i = n − 1` thì `colors[n]` tràn — nhưng chỉ khi hai ký tự cuối bằng nhau (so sánh chuỗi `==` dừng sớm), nên lỗi lúc có lúc không. **Sửa:** `range(1, n − 1)`.

### Lỗi 6 — Mô phỏng thật bằng cắt chuỗi cho chuỗi dài

In [24]:
def simulate_slicing(colors):
    s, turn = colors, "A"
    while True:
        m = s.find(turn * 3)
        if m == -1:
            return turn == "B"
        s = s[:m + 1] + s[m + 2:]; turn = "B" if turn == "A" else "A"

for n in (1000, 4000, 16000):
    s = "A" * n + "B" * n
    t = time.perf_counter(); r = simulate_slicing(s); dt = time.perf_counter() - t
    print(f"n={2 * n:>6}: mô phỏng {dt * 1000:6.0f} ms -> {r} | đếm: {Solution().winnerOfGame(s)}")

n=  2000: mô phỏng      1 ms -> False | đếm: False
n=  8000: mô phỏng      6 ms -> False | đếm: False
n= 32000: mô phỏng     86 ms -> False | đếm: False


**Vì sao không tốt:** mỗi nước tạo chuỗi mới `O(n)` và có `O(n)` nước ⇒ `O(n²)`; thời gian tăng khoảng 4 lần khi `n` gấp đôi. **Sửa:** đếm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài trò chơi có `n` lớn (`10⁵`) ⇒ gần như chắc chắn có **công thức**/bất biến, không phải minimax.
- Kiểm tra xem nước đi của hai bên có **ảnh hưởng nhau** không; nếu không, mỗi người có một "kho" độc lập ⇒ so sánh kích thước kho (người đi trước cần **nhiều hơn hẳn**).
- Đếm mẫu chồng lấn: duyệt vị trí, hoặc regex lookahead `(?=...)`; tránh `str.count`.
- Viết minimax nhỏ để xác nhận giả thuyết trước khi tin công thức.

## 15. Biến thể của bài

1. **292 Nim Game**, **1025 Divisor Game**: trò chơi có công thức đơn giản (`n % 4`, tính chẵn lẻ).
2. **Bob đi trước** ⇒ Alice thắng ⇔ `a ≥ b`.
3. **Cho phép xóa cả quân ở mép** nếu hàng xóm duy nhất cùng màu ⇒ loạt dài `L` cho `L − 1` nước (trừ loạt chiếm cả hàng) — Bài tập 5.
4. **Ba màu / `k` người chơi** lần lượt — cùng ý tưởng kho độc lập, người hết kho đầu tiên thua.
5. **877 Stone Game / 1140**: các nước **có** ảnh hưởng nhau ⇒ cần DP minimax thật sự.

### 15.1 Code biến thể: `k` người chơi với `k` màu, kiểm bằng minimax

In [25]:
def k_player_loser(colors, order):
    # order: danh sách màu theo thứ tự lượt; trả về màu của người thua đầu tiên
    stock = {c: triples(colors, c) for c in order}
    t = 0
    while True:
        c = order[t % len(order)]
        if stock[c] == 0:
            return c
        stock[c] -= 1; t += 1

def k_player_brute(colors, order):
    @lru_cache(maxsize=None)
    def loser(s, t):
        c = order[t % len(order)]
        moves = [i for i in range(1, len(s) - 1) if s[i - 1] == s[i] == s[i + 1] == c]
        if not moves:
            return c
        return loser(s[:moves[0]] + s[moves[0] + 1:], t + 1)   # mọi nước tương đương nên lấy nước đầu
    return loser(colors, 0)

rng = random.Random(15)
for _ in range(500):
    s = "".join(rng.choice("ABC") for _ in range(rng.randint(1, 20)))
    assert k_player_loser(s, "ABC") == k_player_brute(s, "ABC")
print("3 người chơi: người thua theo kho = mô phỏng (500 chuỗi);", k_player_loser("AAAABBBCCCCC", "ABC"))

3 người chơi: người thua theo kho = mô phỏng (500 chuỗi); B


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Minimax | Duyệt cây trò chơi | mũ | mũ | Kiểm chứng `n ≤ 14` |
| 2. Theo loạt | `max(0, L−2)` mỗi loạt | `O(n)` | `O(1)` | Dễ giải thích |
| 3. Đếm bộ ba | Một lượt | `O(n)` | `O(1)` | **Bản nộp** |
| 4. Regex / `zip` | Lookahead chồng lấn | `O(n)` | `O(n)` | Viết một dòng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `"AAAA"`; (b) `"BBBAAA"`; (c) `"AAAABBBB"`; (d) `"ABBBA"`.

**Bài 2 (tính tay):** Với `"AAAAABBAAAABBBBBB"`, liệt kê các loạt, tính `a`, `b` và chơi thử từng lượt đến khi có người thua.

**Bài 3 (điền TODO):** Hoàn thiện `count_moves(colors)` trả về `(a, b)` theo loạt.

**Bài 4 (tự viết lại):** Viết `winner_bob_first(colors)` cho biến thể Bob đi trước; đối chiếu minimax.

**Bài 5 (mini challenge):** Biến thể: được xóa cả quân **ở mép** nếu **hàng xóm duy nhất** của nó cùng màu (quân giữa vẫn cần hai hàng xóm cùng màu). Viết công thức `winner_edge(colors)` và kiểm chứng bằng minimax — chú ý tính độc lập có còn đúng không.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_moves(colors):
    a = b = 0
    for ch, g in groupby(colors):
        # TODO: L = độ dài loạt; cộng max(0, L - 2) vào a hoặc b theo ch
        pass
    return a, b

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def winner_bob_first(colors):
    # TODO: Bob đi trước -> Alice thắng khi Bob hết kho trước: a >= b
    pass

def winner_edge(colors):
    # TODO: quân mép xóa được nếu hàng xóm cùng màu; thử viết minimax trước, rồi tìm công thức theo loạt
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đếm bộ ba. Mức 2: so `a > b`. Mức 3: (a) `a = 2, b = 0` ⇒ `True`; (b) `1, 1` ⇒ `False`; (c) `2, 2` ⇒ `False`; (d) `0, 1` ⇒ `False`.

**Bài 2.** Mức 1: loạt `A5, B2, A4, B6`. Mức 2: `a = 3 + 2 = 5`, `b = 0 + 4 = 4`. Mức 3: sau 4 vòng (Alice 4, Bob 4) kho là `(1, 0)`; Alice đi nước thứ 5, Bob hết nước ⇒ `True`.

**Bài 3.** Mức 1: `L = sum(1 for _ in g)`. Mức 2: chỉ khi `L ≥ 3`. Mức 3: trả tuple.

**Bài 4.** Mức 1: Bob thua ở lượt `2b + 1`, Alice ở lượt `2a + 2`. Mức 2: Alice thắng ⇔ `2b + 1 < 2a + 2` ⇔ `b ≤ a`. Mức 3: viết minimax với lượt đầu là `'B'`.

**Bài 5.** Mức 1: quân mép của loạt chạm mép hàng xóa được khi loạt dài `≥ 2`. Mức 2: xóa quân mép không làm hai loạt khác màu nối nhau ⇒ vẫn độc lập. Mức 3: loạt dài `L` cho `L − 2 + (số đầu loạt chạm mép hàng)` nước, nhưng loạt dài 1 ở mép không xóa được (không có hàng xóm cùng màu) và loạt chiếm **cả hàng** dài `L` cho `L − 1` (quân cuối cùng không có hàng xóm). Kiểm bằng minimax.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2, 3
f = Solution().winnerOfGame
b1 = [f(s) for s in ("AAAA", "BBBAAA", "AAAABBBB", "ABBBA")]
print("Bài 1:", b1); assert b1 == [True, False, False, False]
s2 = "AAAAABBAAAABBBBBB"
print("Bài 2: loạt", [(c, len(list(g))) for c, g in groupby(s2)], "a,b =", triples(s2, "A"), triples(s2, "B"))
assert show_game(s2) is True and winner_runs(s2) is True
def count_moves(colors):
    a = b = 0
    for ch, g in groupby(colors):
        L = sum(1 for _ in g)
        if L >= 3:
            if ch == "A": a += L - 2
            else: b += L - 2
    return a, b
rng = random.Random(3)
for _ in range(500):
    s = "".join(rng.choice("AB") for _ in range(rng.randint(1, 25)))
    assert count_moves(s) == (triples(s, "A"), triples(s, "B"))
print("Bài 3: OK")

Bài 1: [True, False, False, False]
Bài 2: loạt [('A', 5), ('B', 2), ('A', 4), ('B', 6)] a,b = 5 4
lượt 1: Alice đi, kho còn A=4, B=4
lượt 2: Bob đi, kho còn A=4, B=3
lượt 3: Alice đi, kho còn A=3, B=3
lượt 4: Bob đi, kho còn A=3, B=2
lượt 5: Alice đi, kho còn A=2, B=2
lượt 6: Bob đi, kho còn A=2, B=1
lượt 7: Alice đi, kho còn A=1, B=1
lượt 8: Bob đi, kho còn A=1, B=0
lượt 9: Alice đi, kho còn A=0, B=0
lượt 10: Bob hết nước -> Alice thắng
Bài 3: OK


In [29]:
# Bài 4, 5
def winner_bob_first(colors):
    a, b = count_moves(colors)
    return a >= b

def minimax_general(colors, first, can_remove):
    @lru_cache(maxsize=None)
    def win(s, ch):
        other = "B" if ch == "A" else "A"
        for i in range(len(s)):
            if s[i] == ch and can_remove(s, i) and not win(s[:i] + s[i + 1:], other):
                return True
        return False
    return win(colors, first)

def mid_rule(s, i):
    return 0 < i < len(s) - 1 and s[i - 1] == s[i] == s[i + 1]

def edge_rule(s, i):
    if len(s) == 1: return False
    if i == 0: return s[1] == s[0]
    if i == len(s) - 1: return s[-2] == s[-1]
    return s[i - 1] == s[i] == s[i + 1]

def winner_edge(colors):
    n, cnt = len(colors), {"A": 0, "B": 0}
    pos = 0
    for ch, g in groupby(colors):
        L = sum(1 for _ in g)
        if L == n:
            cnt[ch] += L - 1
        else:
            touches = (pos == 0) + (pos + L == n)
            cnt[ch] += max(0, L - 2) + (touches if L >= 2 else 0)
        pos += L
    return cnt["A"] > cnt["B"]

for n in range(1, 11):
    for bits in product("AB", repeat=n):
        s = "".join(bits)
        assert winner_bob_first(s) == (not minimax_general(s, "B", mid_rule))
        assert winner_edge(s) == minimax_general(s, "A", edge_rule), s
print("Bài 4, 5: OK trên mọi chuỗi độ dài ≤ 10 |", winner_bob_first("AAABBB"), winner_edge("AAB"), winner_edge("AABB"))

Bài 4, 5: OK trên mọi chuỗi độ dài ≤ 10 | True True False


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
a = b = 0
for i in range(1, len(s) - 1):
    if s[i - 1] == s[i] == s[i + 1]:
        if s[i] == "A": a += 1
        else: b += 1
return a > b            # Alice đi trước: cần NHIỀU HƠN HẲN
```

**Checklist:**
- [ ] Đã xác nhận nước đi hai bên **độc lập** (không cần minimax)?
- [ ] Loạt dài `L` cho `L − 2` nước (không phải `L − 1`)?
- [ ] Đếm **chồng lấn** (không dùng `str.count`)?
- [ ] So sánh `a > b` (không phải `>=`)?
- [ ] Vòng lặp `range(1, n − 1)` (tránh `s[−1]` vòng và tràn chỉ số)?